# MPX U-Net Training with Balanced Patch Set

- v1: L1PO on balanced patch set (1000 pateches per patient, N=19)

- JIDisease_v3: Removed patient 249 due to high severity and large number of coalesced lesions.

Last update: 220601 AMcNeil

In [31]:
# Base oackages
import numpy as np
import matplotlib.pyplot as plt
import os

# I/O
import glob
from skimage import io
import h5py
import json

# Pytorch
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import ToTensor

# Segmentation Models
import segmentation_models_pytorch as smp
import segmentation_models_pytorch.utils as smputil

# Data Augmentation
import imgaug as ia
import imgaug.augmenters as iaa
from imgaug.augmentables.segmaps import SegmentationMapsOnImage
ia.seed(2)

random_state = np.random.RandomState(42)

In [4]:
torch.cuda.empty_cache()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [5]:
def path_join(folder_list):
    # Creates folder path from input list, including drive name
    k=0
    folder_path = str()
    for directory in folder_list:
        if k == 0:
            if len(directory) == 1 and directory != '.':
                folder_path += directory + ':' + os.sep
            else:
                folder_path += directory + os.sep
        else:
            folder_path += directory + os.sep
        k+=1
    
    return folder_path

---
## Set variables for data locations, names, etc.

In [22]:
# Data folder (laptop)
data_root = 'MPX_JAMADerm_TrainingPatches/'

print('Data root location set to: \"'+data_root+'\"' )
#img_list = sorted(glob.glob(data_root + '**' + os.sep + '*_img_*.png', recursive=True))
#gt_list = sorted(glob.glob(data_root + '**' + os.sep + '*_gt_*.png', recursive=True))
#img_list = glob.glob(img_dir + '*/*.png')

Data root location set to: "MPX_JAMADerm_TrainingPatches/"


In [23]:
dir_list = sorted(glob.glob(data_root + '*'))

img_paths_dict = {}
gt_paths_dict = {}
img_name_list = []
for directory in dir_list:
    img_name = os.path.basename(directory)
    img_name_list.extend([img_name])
    
    img_paths_dict[img_name] = sorted(glob.glob(directory +os.sep+ '*_img_*.png'))
    gt_paths_dict[img_name] = sorted(glob.glob(directory +os.sep+ '*_gt_*.png'))

In [24]:
dir_list

['MPX_JAMADerm_TrainingPatches\\051301',
 'MPX_JAMADerm_TrainingPatches\\051302',
 'MPX_JAMADerm_TrainingPatches\\051307',
 'MPX_JAMADerm_TrainingPatches\\051308',
 'MPX_JAMADerm_TrainingPatches\\051309',
 'MPX_JAMADerm_TrainingPatches\\051310',
 'MPX_JAMADerm_TrainingPatches\\213',
 'MPX_JAMADerm_TrainingPatches\\221',
 'MPX_JAMADerm_TrainingPatches\\227',
 'MPX_JAMADerm_TrainingPatches\\236',
 'MPX_JAMADerm_TrainingPatches\\238',
 'MPX_JAMADerm_TrainingPatches\\239',
 'MPX_JAMADerm_TrainingPatches\\247',
 'MPX_JAMADerm_TrainingPatches\\248',
 'MPX_JAMADerm_TrainingPatches\\Kole1',
 'MPX_JAMADerm_TrainingPatches\\Kole2',
 'MPX_JAMADerm_TrainingPatches\\Kole3',
 'MPX_JAMADerm_TrainingPatches\\MPX subject coming from Dekese (Not enrolled)']

In [25]:
def file_list_dict(img_list, gt_list):
    # Returns a dictionary with the list of images and corresponding ground truth
    file_list = {"imgs": img_list,
      "gt": gt_list}
    
    return file_list

In [26]:
def train_valid_split(img_list, gt_list, random_state, proportion=9):
    train_img_list = []
    train_gt_list = []
    valid_img_list = []
    valid_gt_list = []
    
    valid_select_vector = np.zeros(len(img_list)).astype('int')
    no_of_training_imgs = int(len(img_list)/proportion)
    
    k = 0
    for i in range(no_of_training_imgs):
        foo = random_state.randint(0,proportion)
        valid_select_vector[foo + k] = 1
        k+=proportion
    
    for img_no in range(len(img_list)):
        if valid_select_vector[img_no] == 0:
            train_img_list.extend([img_list[img_no]])
            train_gt_list.extend([gt_list[img_no]])
        else:
            valid_img_list.extend([img_list[img_no]])
            valid_gt_list.extend([gt_list[img_no]])
    
    return train_img_list, train_gt_list, valid_img_list, valid_gt_list

In [27]:
class GVHD_Dataset(Dataset):
    def __init__(self, files, mode, transform=None):
        self.transform = transform 
        self.files = files
        self.mode = mode
        
    def __len__(self):
        size=len(self.files["imgs"])
        return size
    
    def __getitem__(self, idx):
        path_img = self.files["imgs"][idx]
        path_mask = self.files["gt"][idx]
        
        if self.mode == 'test':
            image = path_img
            mask = path_mask
        else:
            image = io.imread(path_img)
            mask = io.imread(path_mask)#[:,:,0]
        
        if self.mode == 'train':
            segmap = SegmentationMapsOnImage(mask, shape=image.shape)
            image_aug, segmap_aug = augmentation_pipeline_train(image=image, segmentation_maps=segmap)
            
            mask = self.transform['mask'](segmap_aug.get_arr())
            image = self.transform['image'](image_aug)
        else:
            mask = self.transform['mask'](mask)
            image = self.transform['image'](image)
        
        image = image.float().to(device)
        mask = mask.float().to(device)
        
        return image, mask

In [28]:
augmentation_pipeline_train = iaa.Sequential([
    iaa.Sometimes(0.5, iaa.ElasticTransformation(alpha=100, sigma=10)),
    iaa.Sometimes(0.5, iaa.Fliplr(1)),
    iaa.Sometimes(0.5, iaa.GaussianBlur(sigma=(0.0, 2.0))),
    iaa.Sometimes(0.5, iaa.Affine(scale=(0.5, 2.0), translate_percent={"x": (-0.15, 0.15), "y": (-0.15, 0.15)}, rotate=(-45, 45))),
    iaa.Sometimes(0.5, iaa.PerspectiveTransform(scale=(0.01, 0.15))),
    iaa.Sometimes(0.5, iaa.ChangeColorTemperature((4500, 9500))),
    iaa.Sometimes(0.5, iaa.GammaContrast((0.75, 1.25)))
])

composed = {
    'image':transforms.Compose([ToTensor()]),
    'mask':transforms.Compose([ToTensor()]),
}

---
# Training GVHD UNet for Monkeypox

In [35]:
def cGVHD_UNet_training(data_train,
                        data_valid,
                        loss=smputil.losses.BCELoss(),
                        ENCODER = 'resnet34',
                        ENCODER_WEIGHTS = 'imagenet',
                        ACTIVATION = 'sigmoid',
                        save_root='.' + os.sep,
                        save_name='cGVHD_UNet_',
                        total_epochs = 50,
                        lr_change_epoch = 25,
                        lr_1 = 0.0001,
                        lr_2 = 0.00001,
                        train_batchsize = 4,
                        valid_batchsize = 1
                       ):
    torch.cuda.empty_cache()

    print('Training ' + save_name + ' with ' + loss.__name__ + ' for ' + str(total_epochs) + ' epochs...')

    train_loader = DataLoader(data_train, batch_size=train_batchsize, shuffle=True)
    valid_loader = DataLoader(data_valid, batch_size=valid_batchsize, shuffle=False)
    
    DEVICE = 'cuda'

    # create segmentation model with pretrained encoder
    model = smp.UnetPlusPlus(
        encoder_name=ENCODER, 
        encoder_weights=ENCODER_WEIGHTS, 
        classes=1, 
        activation=ACTIVATION,
        #encoder_depth=3
    )

    metrics = [
        #smp.utils.metrics.Accuracy(),
        smp.utils.metrics.Fscore(),
        #smp.utils.metrics.IoU(),
        #smp.utils.metrics.Precision(),
        #smp.utils.metrics.Recall()
    ]

    optimizer = torch.optim.Adam([ 
        dict(params=model.parameters(), lr=lr_1),
    ])

    # EPOCHS
    # it is a simple loop of iterating over dataloader`s samples
    train_epoch = smp.utils.train.TrainEpoch(
        model, 
        loss=loss, 
        metrics=metrics, 
        optimizer=optimizer,
        device=DEVICE,
        verbose=True,
    )

    valid_epoch = smp.utils.train.ValidEpoch(
        model, 
        loss=loss, 
        metrics=metrics, 
        device=DEVICE,
        verbose=True,
    )

    # TRAIN
    max_score_fscore = 0
    
    training_performance = {}
    training_performance['loss'] = []
    training_performance['dsc'] = []

    validation_performance = {}
    validation_performance['loss'] = []
    validation_performance['dsc'] = []
    
    savename_head = save_root + save_name + ENCODER + '_' + loss.__name__ + '_'

    for epoch_no in range(total_epochs):
        print('\nEpoch: {}'.format(epoch_no))
        train_logs = train_epoch.run(train_loader)
        valid_logs = valid_epoch.run(valid_loader)

        if max_score_fscore < valid_logs['fscore']:
            max_score_fscore = valid_logs['fscore']
            torch.save(model, savename_head + 'best.pth')
            print('fscore model saved!')

        training_performance['loss'].extend([train_logs['bce_loss']])
        training_performance['dsc'].extend([train_logs['fscore']])
        
        validation_performance['loss'].extend([valid_logs['bce_loss']])
        validation_performance['dsc'].extend([valid_logs['fscore']])

        if epoch_no == lr_change_epoch:
            optimizer.param_groups[0]['lr'] = lr_2
            print('Decrease decoder learning rate to ' + str(lr_2))
        
        with open(savename_head + 'training_performance.json', 'w') as fp:
            json.dump(training_performance, fp,  indent=4)
    
        with open(savename_head + 'validation_performance.json', 'w') as fp:
            json.dump(validation_performance, fp,  indent=4)

In [37]:
train_batchsize = 16

for patient_no in range(0,14):
    patient_name = img_name_list[patient_no]
    print('Patient: ' + str(patient_name))
    
    save_folder = path_join(['.','220601_MPX_JAMADerm_N18_v6_results'])
    if os.path.dirname(save_folder):
        os.makedirs(os.path.dirname(save_folder), exist_ok=True)
    
    save_name = '220601_MPX_JAMADerm_N18_v6_patient-' + patient_name + '_'
    
    img_list = []
    gt_list = []
    for key in img_paths_dict:
        if key is not patient_name:
            img_list.extend(img_paths_dict[key])
            gt_list.extend(gt_paths_dict[key])
    
    # Create train and validation file lists, and prepare data loaders
    train_img_list,train_gt_list,valid_img_list,valid_gt_list = train_valid_split(img_list,gt_list,random_state, 10)
    
    files_train = file_list_dict(train_img_list, train_gt_list)
    dataset_train = GVHD_Dataset(files_train,'train',composed)
    
    files_valid = file_list_dict(valid_img_list, valid_gt_list)
    dataset_valid = GVHD_Dataset(files_valid,'valid',composed)
    
    cGVHD_UNet_training(dataset_train, dataset_valid,
                        save_root=save_folder, save_name=save_name,
                        train_batchsize=train_batchsize,
                        total_epochs = 40,
                        lr_change_epoch = 30
                       )

Patient: 051301
Training 220601_MPX_JAMADerm_N18_v6_patient-051301_ with bce_loss for 40 epochs...

Epoch: 0
valid: 100%|██████████████████████████████████| 1700/1700 [00:43<00:00, 39.31it/s, bce_loss - 0.05145, fscore - 0.6327]
fscore model saved!

Epoch: 1
valid: 100%|█████████████████████████████████| 1700/1700 [00:14<00:00, 116.07it/s, bce_loss - 0.02997, fscore - 0.6739]
fscore model saved!

Epoch: 2
valid: 100%|█████████████████████████████████| 1700/1700 [00:14<00:00, 115.67it/s, bce_loss - 0.02419, fscore - 0.7123]
fscore model saved!

Epoch: 3
valid: 100%|█████████████████████████████████| 1700/1700 [00:14<00:00, 116.07it/s, bce_loss - 0.01999, fscore - 0.7416]
fscore model saved!

Epoch: 4
valid: 100%|█████████████████████████████████| 1700/1700 [00:14<00:00, 116.61it/s, bce_loss - 0.01874, fscore - 0.7767]
fscore model saved!

Epoch: 5
valid: 100%|█████████████████████████████████| 1700/1700 [00:14<00:00, 116.76it/s, bce_loss - 0.01796, fscore - 0.7468]

Epoch: 6
valid: 100%

valid: 100%|████████████████████████████████| 1700/1700 [00:14<00:00, 115.86it/s, bce_loss - 0.009988, fscore - 0.8664]
Decrease decoder learning rate to 1e-05

Epoch: 31
valid: 100%|█████████████████████████████████| 1700/1700 [00:14<00:00, 116.50it/s, bce_loss - 0.00891, fscore - 0.8859]
fscore model saved!

Epoch: 32
valid: 100%|██████████████████████████████████| 1700/1700 [00:43<00:00, 38.72it/s, bce_loss - 0.00874, fscore - 0.8885]
fscore model saved!

Epoch: 33
valid: 100%|████████████████████████████████| 1700/1700 [00:14<00:00, 116.38it/s, bce_loss - 0.008581, fscore - 0.8913]
fscore model saved!

Epoch: 34
valid: 100%|████████████████████████████████| 1700/1700 [00:14<00:00, 116.32it/s, bce_loss - 0.008441, fscore - 0.8943]
fscore model saved!

Epoch: 35
valid: 100%|█████████████████████████████████| 1700/1700 [00:43<00:00, 39.28it/s, bce_loss - 0.008398, fscore - 0.8958]
fscore model saved!

Epoch: 36
valid: 100%|████████████████████████████████| 1700/1700 [00:13<00:00, 123.

valid: 100%|██████████████████████████████████| 1700/1700 [00:40<00:00, 41.75it/s, bce_loss - 0.01193, fscore - 0.8505]
fscore model saved!

Epoch: 21
valid: 100%|██████████████████████████████████| 1700/1700 [00:13<00:00, 121.83it/s, bce_loss - 0.01162, fscore - 0.854]
fscore model saved!

Epoch: 22
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 123.21it/s, bce_loss - 0.02662, fscore - 0.8047]

Epoch: 23
valid: 100%|██████████████████████████████████| 1700/1700 [00:42<00:00, 40.42it/s, bce_loss - 0.01436, fscore - 0.8247]

Epoch: 24
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 123.73it/s, bce_loss - 0.01188, fscore - 0.8433]

Epoch: 25
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 122.91it/s, bce_loss - 0.01065, fscore - 0.8729]
fscore model saved!

Epoch: 26
valid: 100%|██████████████████████████████████| 1700/1700 [00:41<00:00, 40.77it/s, bce_loss - 0.01163, fscore - 0.8537]

Epoch: 27
valid: 100%|███████████

valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 122.34it/s, bce_loss - 0.01425, fscore - 0.8234]
fscore model saved!

Epoch: 12
valid: 100%|██████████████████████████████████| 1700/1700 [00:42<00:00, 40.31it/s, bce_loss - 0.01335, fscore - 0.8419]
fscore model saved!

Epoch: 13
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 123.12it/s, bce_loss - 0.01365, fscore - 0.8313]

Epoch: 14
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 123.03it/s, bce_loss - 0.01305, fscore - 0.8416]

Epoch: 15
valid: 100%|██████████████████████████████████| 1700/1700 [00:17<00:00, 95.72it/s, bce_loss - 0.01317, fscore - 0.8413]

Epoch: 16
valid: 100%|██████████████████████████████████| 1700/1700 [00:37<00:00, 45.80it/s, bce_loss - 0.01232, fscore - 0.8472]
fscore model saved!

Epoch: 17
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 122.03it/s, bce_loss - 0.01197, fscore - 0.8529]
fscore model saved!

Epoch: 18
val

valid: 100%|██████████████████████████████████| 1700/1700 [00:41<00:00, 40.70it/s, bce_loss - 0.03295, fscore - 0.7172]
fscore model saved!

Epoch: 2
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 123.55it/s, bce_loss - 0.02529, fscore - 0.7186]
fscore model saved!

Epoch: 3
valid: 100%|██████████████████████████████████| 1700/1700 [00:14<00:00, 121.13it/s, bce_loss - 0.0229, fscore - 0.7355]
fscore model saved!

Epoch: 4
valid: 100%|██████████████████████████████████| 1700/1700 [00:42<00:00, 40.33it/s, bce_loss - 0.01855, fscore - 0.7751]
fscore model saved!

Epoch: 5
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 121.82it/s, bce_loss - 0.01727, fscore - 0.7945]
fscore model saved!

Epoch: 6
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 122.13it/s, bce_loss - 0.01602, fscore - 0.8064]
fscore model saved!

Epoch: 7
valid: 100%|██████████████████████████████████| 1700/1700 [00:41<00:00, 40.59it/s, bce_loss - 0.0157

...

valid: 100%|████████████████████████████████| 1700/1700 [00:13<00:00, 122.75it/s, bce_loss - 0.009871, fscore - 0.8764]
fscore model saved!

Epoch: 26
valid: 100%|████████████████████████████████| 1700/1700 [00:14<00:00, 120.54it/s, bce_loss - 0.009662, fscore - 0.8809]
fscore model saved!

Epoch: 27
valid: 100%|██████████████████████████████████| 1700/1700 [00:41<00:00, 40.86it/s, bce_loss - 0.01004, fscore - 0.8762]

Epoch: 28
valid: 100%|██████████████████████████████████| 1700/1700 [00:13<00:00, 123.32it/s, bce_loss - 0.00928, fscore - 0.885]
fscore model saved!

Epoch: 29
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 122.81it/s, bce_loss - 0.009921, fscore - 0.876]

Epoch: 30
valid: 100%|█████████████████████████████████| 1700/1700 [00:41<00:00, 41.07it/s, bce_loss - 0.009544, fscore - 0.8855]
fscore model saved!
Decrease decoder learning rate to 1e-05

Epoch: 31
valid: 100%|████████████████████████████████| 1700/1700 [00:13<00:00, 123.29it/s, bce_loss - 0

valid: 100%|███████████████████████████████████| 1700/1700 [00:35<00:00, 47.40it/s, bce_loss - 0.01262, fscore - 0.842]

Epoch: 16
valid: 100%|██████████████████████████████████| 1700/1700 [00:19<00:00, 85.23it/s, bce_loss - 0.01299, fscore - 0.8325]

Epoch: 17
valid: 100%|██████████████████████████████████| 1700/1700 [00:13<00:00, 123.03it/s, bce_loss - 0.01219, fscore - 0.855]
fscore model saved!

Epoch: 18
valid: 100%|█████████████████████████████████| 1700/1700 [00:14<00:00, 121.40it/s, bce_loss - 0.01132, fscore - 0.8597]
fscore model saved!

Epoch: 19
valid: 100%|██████████████████████████████████| 1700/1700 [00:42<00:00, 40.08it/s, bce_loss - 0.01139, fscore - 0.8585]

Epoch: 20
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 122.25it/s, bce_loss - 0.01086, fscore - 0.8621]
fscore model saved!

Epoch: 21
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 122.37it/s, bce_loss - 0.01119, fscore - 0.8581]

Epoch: 22
valid: 100%|███████████

valid: 100%|██████████████████████████████████| 1700/1700 [00:14<00:00, 117.40it/s, bce_loss - 0.0171, fscore - 0.7881]
fscore model saved!

Epoch: 6
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 123.22it/s, bce_loss - 0.01772, fscore - 0.7538]

Epoch: 7
valid: 100%|██████████████████████████████████| 1700/1700 [00:42<00:00, 40.34it/s, bce_loss - 0.01646, fscore - 0.7932]
fscore model saved!

Epoch: 8
valid: 100%|█████████████████████████████████| 1700/1700 [00:14<00:00, 121.30it/s, bce_loss - 0.01515, fscore - 0.8076]
fscore model saved!

Epoch: 9
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 122.86it/s, bce_loss - 0.01544, fscore - 0.8163]
fscore model saved!

Epoch: 10
valid: 100%|██████████████████████████████████| 1700/1700 [00:41<00:00, 40.77it/s, bce_loss - 0.01372, fscore - 0.8264]
fscore model saved!

Epoch: 11
valid: 100%|█████████████████████████████████| 1700/1700 [00:14<00:00, 120.66it/s, bce_loss - 0.01459, fscore - 0.8055

valid: 100%|████████████████████████████████| 1700/1700 [00:13<00:00, 123.41it/s, bce_loss - 0.008696, fscore - 0.8949]

Epoch: 37
valid: 100%|████████████████████████████████| 1700/1700 [00:13<00:00, 122.00it/s, bce_loss - 0.008548, fscore - 0.8967]
fscore model saved!

Epoch: 38
valid: 100%|████████████████████████████████| 1700/1700 [00:13<00:00, 122.62it/s, bce_loss - 0.008488, fscore - 0.8971]
fscore model saved!

Epoch: 39
valid: 100%|██████████████████████████████████| 1700/1700 [00:41<00:00, 40.62it/s, bce_loss - 0.008551, fscore - 0.895]
Patient: 248
Training 220601_MPX_JAMADerm_N18_v6_patient-248_ with bce_loss for 40 epochs...

Epoch: 0
valid: 100%|██████████████████████████████████| 1700/1700 [00:15<00:00, 109.15it/s, bce_loss - 0.0841, fscore - 0.5642]
fscore model saved!

Epoch: 1
valid: 100%|█████████████████████████████████| 1700/1700 [00:13<00:00, 122.31it/s, bce_loss - 0.03781, fscore - 0.6586]
fscore model saved!

Epoch: 2
valid: 100%|████████████████████████████████

valid: 100%|█████████████████████████████████| 1700/1700 [00:15<00:00, 112.84it/s, bce_loss - 0.01094, fscore - 0.8688]

Epoch: 27
valid: 100%|███████████████████████████████████| 1700/1700 [00:42<00:00, 39.68it/s, bce_loss - 0.0101, fscore - 0.8801]
fscore model saved!

Epoch: 28
valid: 100%|██████████████████████████████████| 1700/1700 [00:14<00:00, 115.34it/s, bce_loss - 0.0103, fscore - 0.8754]

Epoch: 29
valid: 100%|█████████████████████████████████| 1700/1700 [00:14<00:00, 113.65it/s, bce_loss - 0.01028, fscore - 0.8768]

Epoch: 30
valid: 100%|██████████████████████████████████| 1700/1700 [00:43<00:00, 39.35it/s, bce_loss - 0.01008, fscore - 0.8751]
Decrease decoder learning rate to 1e-05

Epoch: 31
valid: 100%|████████████████████████████████| 1700/1700 [00:14<00:00, 114.10it/s, bce_loss - 0.009202, fscore - 0.8894]
fscore model saved!

Epoch: 32
valid: 100%|████████████████████████████████| 1700/1700 [00:14<00:00, 114.34it/s, bce_loss - 0.009026, fscore - 0.8919]
fscore model s

In [22]:
for i in range (0,9):
    print(i)

0
1
2
3
4
5
6
7
8


In [23]:
for i in range (9,14):
    print(i)

9
10
11
12
13


In [24]:
for i in range (14,18):
    print(i)

14
15
16
17
